In [1]:
from actor_system import ActorSystem, PipelinedActor
from msg_type import CellFound, DataReady, BufferRelease, NoCell
import numpy as np

In [2]:
class CellSearchActor(PipelinedActor):
    """Initial search -> tracking on detection. Safe slot resize."""

    def __init__(self, system, buf, pool, graph, params, node_pipeline):
        super().__init__('cell_search', system, pool, graph, node_pipeline[0])
        self._buf = buf
        self._mode = 'initial_search'
        self._track_size = params.pss_tracking_len

    def _default_behavior(self, msg):
        if isinstance(msg, DataReady):
            if self._mode == 'tracking' and msg.tag.get('mode') == 'initial_search':
                return
        super()._default_behavior(msg)

    def _fill_slot(self, slot_idx, msg):
        slot = self._pool.slots[slot_idx]
        data = self._buf.read_protected(msg.pid)
        slot.data[:len(data)] = data
        slot.tag = msg.tag
        self.system.send_message('buffer_manager', BufferRelease(pid=msg.pid))

    def on_result(self, msg):
        if self._mode == 'tracking' and msg.tag.get('mode') == 'initial_search':
            return
        slot = self._pool.slots[msg.slot]
        if slot.sss_detected:
            self.system.send_message('controller', CellFound(
                N_id=3 * slot.N_id_1 + slot.N_id_2,
                f_d=slot.f_d, F=slot.F,
                pss_local_index=slot.pss_local_index,
                tag=msg.tag))
            if self._mode == 'initial_search':
                self._switch_to_tracking()
        else:
            self.system.send_message('controller', NoCell(tag=msg.tag))

    def _switch_to_tracking(self):
        self._mode = 'tracking'
        self._queue.clear()
        self._pool.set_size(self._track_size)

In [3]:
class CellSearchSlot:
    """One working slot. Same field interface as PSSChunk so existing
    PSS/SSS functions can read/write without modification.
    """

    def __init__(self, slot_size):
        self.data = np.empty(slot_size, dtype=complex)
        self.tag = None

        # PSS results
        self.rx_norms = None
        self.pss_detected = False
        self.pss_local_index = None
        self.N_id_2 = None

        # SSS results
        self.sss_detected = False   
        self.N_id_1 = None
        self.F = None
        self.f_d = None

    def reset(self):
        self.tag = None
        self.rx_norms = None
        self.pss_detected = False
        self.pss_local_index = None
        self.N_id_2 = None
        self.sss_detected = False
        self.N_id_1 = None
        self.F = None
        self.f_d = None

#### Test
1. cell search pipeline actor + pipeline

In [4]:
from data.lte_system_info import LTEParams
from flow_graph import FlowGraph, FunctionNode, ResizableSlotPool
from cell_search import PSSDetection, SSSDetection
from protected_circular_buffer import CircularBuffer
import time
from actor_system import Actor
from msg_type import PipelineDone

In [5]:
rxf = np.load('data/rx_preprocessed.npy')
Fs = float(np.load('data/Fs.npy'))
params = LTEParams(Fs=Fs)

In [7]:
class CollectorActor(Actor):
    def __init__(self, name, system):
        super().__init__(name, system)
        self.messages = []
    def _default_behavior(self, message):
        self.messages.append(message)

buf = CircularBuffer(len(rxf))
buf.write(rxf)

system = ActorSystem()

pool = ResizableSlotPool(6, lambda: CellSearchSlot(params.N_subframe))

pss_func = PSSDetection(params, peak_ratio=5.0)
sss_func = SSSDetection(params, peak_ratio=8.0)

graph = FlowGraph(num_workers=4)
pss_node = FunctionNode('pss', pool.make_stage(pss_func), concurrency=1)
sss_node = FunctionNode('sss', pool.make_stage(sss_func), concurrency=1,
                        done_callback=lambda token: system.send_message(
                            'cell_search', PipelineDone(slot=token.slot, tag=token.tag)))
graph.add_edge(pss_node, sss_node)

ctrl = CollectorActor('controller', system)
bm = CollectorActor('buffer_manager', system)
cs = CellSearchActor(system, buf, pool, graph, params, [pss_node, sss_node])

system.create_actor(ctrl)
system.create_actor(cs)
system.create_actor(bm)

# ---- initial search: first half-frame ----

stride = params.stride
pos = 0
chunk_id = 0
while pos + params.N_subframe <= len(rxf) and pos < params.N_half_frame:
    pid = buf.protect(pos, params.N_subframe)
    system.send_message('cell_search', DataReady(
        pid=pid, tag={'mode': 'initial_search', 'chunk_tag': chunk_id, 'pos': pos}))
    pos += stride
    chunk_id += 1

time.sleep(0.5)

# ---- check initial search result ----
cell_found = [m for m in ctrl.messages if isinstance(m, CellFound)]
no_cell = [m for m in ctrl.messages if isinstance(m, NoCell)]
releases = [m for m in bm.messages if isinstance(m, BufferRelease)]

print(f'Sent {chunk_id} initial search chunks')
print(f'Controller: {len(cell_found)} CellFound, {len(no_cell)} NoCell')
print(f'Buffer releases: {len(releases)}')

if cell_found:
    msg = cell_found[0]
    pss_global = msg.tag['pos'] + msg.pss_local_index
    print(f'Initial: PCI={msg.N_id}, F={msg.F}, f_d={msg.f_d:.1f}, pss_global={pss_global}')

    # ---- tracking: one half-frame later ----
    expected_pss = pss_global + params.N_half_frame
    track_start = expected_pss - 2 * params.N_ofdm_sym
    track_len = params.pss_tracking_len

    pid = buf.protect(track_start, track_len)
    system.send_message('cell_search', DataReady(
        pid=pid, tag={'mode': 'tracking', 'frame_tag': 0}))

    time.sleep(0.5)

    cell_found_2 = [m for m in ctrl.messages if isinstance(m, CellFound)]
    print(f'\nAfter tracking: {len(cell_found_2)} CellFound total')
    if len(cell_found_2) > 1:
        r = cell_found_2[1]
        pss_g2 = track_start + r.pss_local_index
        print(f'Tracking: PCI={r.N_id}, F={r.F}, f_d={r.f_d:.1f}, pss_global={pss_g2}')

graph.shutdown()

Sent 6 initial search chunks
Controller: 1 CellFound, 2 NoCell
Buffer releases: 6
Initial: PCI=380, F=0, f_d=1126.9, pss_global=36043

After tracking: 2 CellFound total
Tracking: PCI=380, F=1, f_d=1128.4, pss_global=112842


2. buffer manager + cell search pipeline actor + pipeline 

In [10]:
from buffer_manager_actor import BufferManagerActor
from msg_type import BufferRead

In [11]:
system = ActorSystem()
graph = FlowGraph(num_workers=4)

pool = ResizableSlotPool(6, lambda: CellSearchSlot(params.N_subframe))

pss_func = PSSDetection(params, peak_ratio=5.0)
sss_func = SSSDetection(params, peak_ratio=8.0)

pss_node = FunctionNode('pss', pool.make_stage(pss_func), concurrency=1)
sss_node = FunctionNode('sss', pool.make_stage(sss_func), concurrency=1,
                        done_callback=lambda token: system.send_message(
                            'cell_search', PipelineDone(slot=token.slot, tag=token.tag)))
graph.add_edge(pss_node, sss_node)

ctrl = CollectorActor('controller', system)
bm = BufferManagerActor(system, rxf, buffer_size=params.N_frame,
                        batch_size=params.N_subframe, ingest_delay=0.001)
cs = CellSearchActor(system, bm.buf, pool, graph, params, [pss_node, sss_node])

system.create_actor(ctrl)
system.create_actor(cs)
system.create_actor(bm)

system.send_message('buffer_manager', 'start')

# ---- initial search ----
stride = params.stride
pos = 0
chunk_id = 0
while pos + params.N_subframe <= params.N_half_frame:
    system.send_message('buffer_manager', BufferRead(
        offset=pos, length=params.N_subframe, dest='cell_search',
        tag={'mode': 'initial_search', 'chunk_tag': chunk_id, 'pos': pos}))
    pos += stride
    chunk_id += 1

time.sleep(0.5)

# ---- check initial result ----
cell_found = [m for m in ctrl.messages if isinstance(m, CellFound)]
msg = cell_found[0]
pss_global = msg.tag['pos'] + msg.pss_local_index

# ---- tracking: one half-frame later ----
expected_pss = pss_global + params.N_half_frame
track_start = expected_pss - 2 * params.N_ofdm_sym
track_len = params.pss_tracking_len
track_offset = track_start - bm.buf._abs_read

system.send_message('buffer_manager', BufferRead(
    offset=track_offset, length=track_len, dest='cell_search',
    tag={'mode': 'tracking', 'frame_tag': 0}))

time.sleep(0.5)

# ---- print results ----
print(f'\nController received {len(ctrl.messages)} messages:')
for m in ctrl.messages:
    if isinstance(m, CellFound):
        if m.tag.get('mode') == 'initial_search':
            pss_g = m.tag['pos'] + m.pss_local_index
            print(f'  CellFound (initial): PCI={m.N_id}, F={m.F}, f_d={m.f_d:.1f}, pss_global={pss_g}')
        elif m.tag.get('mode') == 'tracking':
            pss_g = track_start + m.pss_local_index
            print(f'  CellFound (tracking): PCI={m.N_id}, F={m.F}, f_d={m.f_d:.1f}, pss_global={pss_g}')
    elif isinstance(m, NoCell):
        print(f'  NoCell: {m.tag.get("mode")}')
    else:
        print(f'  {m}')

graph.shutdown()


Controller received 5 messages:
  start
  NoCell: initial_search
  NoCell: initial_search
  CellFound (initial): PCI=380, F=0, f_d=1126.9, pss_global=36043
  CellFound (tracking): PCI=380, F=1, f_d=1128.4, pss_global=112842
